# Smart Control Course Project

This notebook runs the maintained Python project modules. It does not contain separate copies of the optimizers or controllers.

**Team:** Anas, Abdullah, Khaira, Kareem, Fares, Fayad, Bahnacy.

The main comparison contains DE, GA, PSO, and SA × four controllers × three plant orders (48 experiments). The optional GP optimizer is run only with the GP controller (three additional experiments).

## 1. Install the permitted dependencies
Run this cell in a fresh Colab runtime.

In [ ]:
%pip -q install control numpy matplotlib

## 2. Upload and extract the project ZIP

Upload the final `Smart-Control-Project.zip` created from the project folder. The code locates the directory containing `smart_control/main.py`, even if the ZIP has one enclosing folder.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
import os
import sys

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if not zip_names:
    raise ValueError('Please upload the final project ZIP file.')

extract_dir = Path('/content/smart_control_project_upload')
extract_dir.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_names[0]) as archive:
    archive.extractall(extract_dir)

main_files = [p for p in extract_dir.rglob('main.py') if p.parent.name == 'smart_control']
project_roots = [p.parent.parent for p in main_files]
if not project_roots:
    raise FileNotFoundError('Could not find smart_control/main.py inside the uploaded ZIP.')
project_root = project_roots[0]
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print('Project root:', project_root)
print('Python source files:', len(list(project_root.rglob('*.py'))))

## 3. Review the experiment configuration

The current entry point uses a low evaluation budget to keep the full matrix within the course runtime target. These settings are a practical starting point, not evidence that every optimizer has converged. The team should agree on the objective weights before claiming that all five metrics are optimized.

In [ ]:
import smart_control.config as config

print('Simulation:', config.SIMULATION)
from smart_control.evaluation.objective import DEFAULT_WEIGHTS
effective_weights = dict(DEFAULT_WEIGHTS)
effective_weights.update(config.OBJECTIVE_WEIGHTS)
print('Effective objective weights:', effective_weights)
print('Plant names:', config.PLANT_NAMES)
print('NN architecture:', config.NN_CONFIG)

## 4. Run the required 48-experiment matrix
This runs DE, GA, PSO, and SA against PID, Neural Network, Fuzzy, and GP controllers on all three plants. Results and plots are saved in the project's `results/` directory.

In [ ]:
from smart_control.main import main

records = main(output_dir=project_root / 'results')
successful = sum(record['status'] == 'success' for record in records)
failed = len(records) - successful
print(f'Completed: {successful}/{len(records)}; failed: {failed}')

## 5. Optional: include the Genetic Programming optimizer
The current GP optimizer evolves GP expression trees, so it is compatible with the GP controller's tree encoding. It is not currently a general-purpose optimizer for PID, NN, or Fuzzy parameter vectors. This optional cell adds three GP-optimizer/GP-controller runs, one per plant.

In [ ]:
# Uncomment to run the three additional GP-optimizer experiments.
# gp_records = main(output_dir=project_root / 'results_with_gp', include_gp=True)
# print(f'GP-inclusive records: {len(gp_records)}')

## 6. Download the generated results
The output folder contains the CSV comparison table and per-plant step-response and convergence plots.

In [ ]:
import shutil

results_dir = project_root / 'results'
if not results_dir.exists():
    raise FileNotFoundError('The results folder was not created. Run the experiment cell first.')
archive_path = shutil.make_archive('/content/smart_control_results', 'zip', results_dir)
print('Created:', archive_path)
files.download(archive_path)

## 7. Before submitting

- Confirm the plant transfer functions and controller design choices with the team/instructor.
- Agree on normalization and weights for all five performance criteria. Current defaults give zero weight to rise time; steady-state error has a default weight of 1.0.
- Check failed runs and inspect the fourth-order plant responses; a successful execution status does not mean the response is well controlled.
- Record the actual runtime from the output. Do not claim the five-minute limit without measuring it in the final Colab environment.
- Explain the low evaluation budget and any limitations in the presentation.